In [1]:
# Imports and Settings
import google.generativeai as genai
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import date
from IPython.display import Markdown
from PIL import Image
from tkinter import Tk
from tkinter.filedialog import askopenfilename

In [2]:
# Set up Gemini AI
genai.configure(api_key=os.environ["GEMINI_API_KEY"]) # API Key is stored in Environment Variables
model = genai.GenerativeModel("gemini-1.5-flash")

In [3]:
# Set up global variables
dates = {}
target_cal = 0
target_pro = 0
target_carb = 0
target_fat = 0

In [4]:
# Function to initialize global target variables
def init(sex, weight, height, age):
    global target_cal
    global target_pro
    global target_carb
    global target_fat
    # Using Mifflin-St Jeor formula (using kg and cm), and moderately active activity level (1.55x)
    if sex == "male":
        target_cal = (10 * weight + 6.25 * height - 5 * age + 5) * 1.55
    else:
        target_cal = (10 * weight + 6.25 * height - 5 * age - 161) * 1.55
    # Using composition that calories should come from 25% protein, 50% carbohydrates, and 25% fat. Then convert to grams
    target_pro = (0.25 * target_cal) / 4
    target_carb = (0.5 * target_cal) / 4
    target_fat = (0.25 * target_cal) / 9

In [5]:
# Function to open user-specified image
def choose_image():
    Tk().withdraw()
    filename = askopenfilename()
    img = Image.open(filename)
    # Downsize image for efficiency
    img.thumbnail((300, 300), Image.LANCZOS) #LANCZOS is more intensive but results in better image quality
    return img

In [6]:
# Function to prompt Gemini
def prompt_gemini(information, img):
    # Build question (string)
    instructions = "Using the information provided, estimate the quantity, calories, protein, carbohydrates, and fat of each food item."
    rules = [
        "Be descriptive.",
        "If there are multiple of the same food, combine them. For example, write 2 buns instead of writing 1 bun twice.",
        #"Prioritise nutritional data from https://fdc.nal.usda.gov.",
        "Write in this format: # of [food], # cal, # g protein, # g carbohydrate, # g fat.",
        "Give a name for the dish on the first line.",
        "Write each ingredient on a new line.",
        "Use metric measurements.",
        "Round to the nearest whole number.",
        "Do not ask the user for more information.",
        "Do not write a disclaimer.",
        "Do not write a total.",
    ]
    string = "Information: "+information+"\nInstructions: "+instructions+"\nRules:"
    for i in rules:
        string += " "+i
    
    # Ask Gemini
    if img is None:
        return model.generate_content(string).text
    else:
        return model.generate_content([string, img]).text

In [7]:
# Class to store and track data for a day
class Entry:
    def __init__(self):
        # Copy global targets into a field in case globals change in the future
        self.target_cal = target_cal
        self.target_pro = target_pro
        self.target_carb = target_carb
        self.target_fat = target_fat
        self.current_cal = 0
        self.current_pro = 0
        self.current_carb = 0
        self.current_fat = 0
        self.meals = []

    # Returns the progress of the current metrics to the target as a percentage, using all meals combined
    def get_total_progress(self):
        return [
            self.current_cal / self.target_cal * 100,
            self.current_pro / self.target_pro * 100,
            self.current_carb / self.target_carb * 100,
            self.current_fat / self.target_fat * 100,
        ]

    # Returns the progress of the current metrics to the target as a percentage, showing the contribution of each meal
    def get_split_progress(self):
        arr = []
        for meal in self.meals:
            arr.append(
                [
                    meal.get_cal() / self.target_cal * 100,
                    meal.get_pro() / self.target_pro * 100,
                    meal.get_carb() / self.target_carb * 100,
                    meal.get_fat() / self.target_fat * 100,
                ]
            )
        return arr
    
    # Returns the names of all stored meals
    def get_meal_names(self):
        arr = []
        for meal in self.meals:
            arr.append(meal.get_name())
        return arr

    # Add a meal to the array and update metrics
    def update(self, meal):
        self.meals.append(meal)
        self.current_cal += meal.get_cal()
        self.current_pro += meal.get_pro()
        self.current_carb += meal.get_carb()
        self.current_fat += meal.get_fat()

    # Print all meals
    def __str__(self):
        string = ""
        for meal in self.meals:
            string += str(meal)
        return string

In [8]:
# Class to consolodate ingredients (items) into a meal
class Meal:
    def __init__(self, name, item, cal, pro, carb, fat):
        self.name = name
        self.item = item
        self.cal = cal
        self.pro = pro
        self.carb = carb
        self.fat = fat

    # Returns only the name of the meal
    def get_name(self):
        return self.name

    # Getter functions to sum all the ingredients' metrics
    def get_cal(self):
        return sum(self.cal)

    def get_pro(self):
        return sum(self.pro)

    def get_carb(self):
        return sum(self.carb)

    def get_fat(self):
        return sum(self.fat)

    # Print meal name with ingredients and their metrics
    def __str__(self):
        string = self.name + "\n"
        for i in range(len(self.item)):
            string += (
                "\t"
                + str(self.item[i])
                + ", "
                + str(self.cal[i])
                + " cal, "
                + str(self.pro[i])
                + " g, "
                + str(self.carb[i])
                + " g, "
                + str(self.fat[i])
                + " g\n"
            )
        string += (
            "\tTotal: "
            + str(self.get_cal())
            + " cal, "
            + str(self.get_pro())
            + " g protein, "
            + str(self.get_carb())
            + " g carbohydrates, "
            + str(self.get_fat())
            + " g fat\n"
        )
        return string

In [9]:
# Parse response
def parse_response(text):
    split = text.split("\n")
    item = []
    cal = []
    pro = []
    carb = []
    fat = []
    # First line is the meal name
    name = split[0]
    # Loop through one ingredient (item) at a time
    for i in split[1:]: 
        # Tokenize the line to seperate each metric
        line = i.split(",")
        # Filter empty lines
        if len(line) == 1:
            continue
        # Add so that an index corresponds to the same ingredient (item) across arrays
        item.append(line[0])
        cal.append(int(line[1].split()[0])) # The number is always the first token
        pro.append(int(line[2].split()[0]))
        carb.append(int(line[3].split()[0]))
        fat.append(int(line[4].split()[0]))

    # Create meal object
    my_meal = Meal(name, item, cal, pro, carb, fat)
    # Store the meal
    today = str(date.today())
    if today not in dates:
        dates[today] = Entry()
    dates[today].update(my_meal)

In [10]:
# Helper function to print all stored meals
def print_all_meals():
    for date in dates:
        print(date)
        print(dates[date])

In [11]:
# Function to graph the progress on the different metrics for a given day
def graph_progress(date):
    # Plot initial bar chart
    x = ["Calories", "Protein", "Carbohydrates", "Fat"]
    y = dates[date].get_split_progress()
    plt.bar(x, y[0])
    # Plot stacked bars
    bot = y[0]
    for i in range(1, len(y)):
        plt.bar(x, y[i], bottom=bot)
        bot = np.add(bot, y[i])
    plt.legend(dates[date].get_meal_names(), bbox_to_anchor=(1.02, 1)) # Position legend to the top-right of plot
    # Threshold line
    plt.axhline(y=100, color="r")
    # Labels
    plt.xlabel("Metric")
    plt.ylabel("Percentage")
    plt.title("Goals for " + str(date))

In [12]:
# Initialize with user-specified characteristics
init("male", 90, 180, 25) # Example - "Average" Male
print(target_cal, target_pro, target_carb, target_fat)

In [13]:
# Get nutritional information from Gemini using text and image
img = None
# Here is where the user would be able to decide whether they would like to include an image
img = choose_image()
display(img)
text = prompt_gemini("1 slice of bread", img)
print(text)

In [14]:
# Analyse response
parse_response(text)

In [15]:
# Verify data is stored correctly
print_all_meals()

In [16]:
# Graph metrics
graph_progress(str(date.today()))